# Clustering avec CAH (Classification Ascendante Hiérarchique) et K-Means sur le Dataset CreditDataSet

Ce notebook présente une analyse de clustering utilisant la Classification Ascendante Hiérarchique (CAH) et K-Means sur le dataset CreditDataSet.

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from kneed import KneeLocator
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et préparation des données

In [ ]:
# Chargement du dataset CreditDataSet
df = pd.read_csv('CreditDataSet.csv')
print(f"Forme du dataset: {df.shape}")

# Suppression de la colonne CUST_ID
df_model = df.drop('CUST_ID', axis=1).copy()

# Gestion des valeurs manquantes dans MINIMUM_PAYMENTS
payments_mean = np.mean(df_model['PAYMENTS'])
for idx, row in df_model.iterrows():
    if pd.isna(row['MINIMUM_PAYMENTS']):
        if row['PAYMENTS'] == 0:
            df_model.loc[idx, 'MINIMUM_PAYMENTS'] = 0
        elif 0 < row['PAYMENTS'] < payments_mean:
            df_model.loc[idx, 'MINIMUM_PAYMENTS'] = row['PAYMENTS']
        else:
            df_model.loc[idx, 'MINIMUM_PAYMENTS'] = payments_mean

# Suppression des lignes avec des valeurs manquantes restantes
df_model = df_model.dropna().reset_index(drop=True)
print(f"Forme après nettoyage: {df_model.shape}")

# Normalisation des données
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_model), 
    columns=df_model.columns
)
print("Données normalisées avec succès!")

## 2. Classification Ascendante Hiérarchique (CAH)

### 2.1 Présentation de la CAH

La **Classification Ascendante Hiérarchique (CAH)** est une méthode de clustering qui regroupe des individus en fonction de leurs similarités ou dissimilarités. 

Cette méthode est dite **"ascendante"** car elle construit une hiérarchie de regroupements à partir de la fusion successive des individus ou des groupes les plus similaires.

**Étapes principales:**
1. Calculer la matrice de distance entre tous les individus
2. Fusionner les deux individus/clusters les plus proches
3. Mettre à jour la matrice de distance
4. Répéter jusqu'à obtenir un seul cluster
5. Visualiser avec un dendrogramme

In [ ]:
# Calcul de la matrice de linkage avec la méthode de Ward
# Ward minimise la variance intra-cluster (somme des carrés des écarts)
linkage_matrix = linkage(df_scaled, method='ward', metric='euclidean')
print("Matrice de linkage calculée avec succès!")
print(f"Forme de la matrice de linkage: {linkage_matrix.shape}")

### 2.2 Visualisation du dendrogramme

In [ ]:
# Affichage du dendrogramme complet
plt.figure(figsize=(15, 8))
dendrogram(linkage_matrix, 
           truncate_mode='level', 
           p=5,  # Afficher seulement les 5 derniers niveaux pour la lisibilité
           leaf_rotation=90,
           leaf_font_size=8)
plt.title('Dendrogramme de la Classification Ascendante Hiérarchique', fontsize=16, pad=20)
plt.xlabel('Individus ou Clusters', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 Détermination du nombre optimal de clusters avec la CAH

In [ ]:
# Calcul du score de silhouette pour différents nombres de clusters
max_clusters = 10
silhouette_scores_cah = []

for num_clusters in range(2, max_clusters + 1):
    clusters_cah = fcluster(linkage_matrix, num_clusters, criterion='maxclust')
    silhouette_avg = silhouette_score(df_scaled, clusters_cah)
    silhouette_scores_cah.append(silhouette_avg)

# Visualisation du score de silhouette
plt.figure(figsize=(10, 6))
plt.plot(range(2, max_clusters + 1), silhouette_scores_cah, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nombre de clusters', fontsize=12)
plt.ylabel('Score de silhouette moyen', fontsize=12)
plt.title('Score de silhouette pour différents nombres de clusters (CAH)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# Trouver le nombre optimal de clusters
optimal_num_clusters_cah = np.argmax(silhouette_scores_cah) + 2
print(f"Nombre optimal de clusters selon le score de silhouette: {optimal_num_clusters_cah}")
print(f"Score de silhouette optimal: {max(silhouette_scores_cah):.4f}")

In [ ]:
# Application de la CAH avec le nombre optimal de clusters
num_clusters_cah = 3  # Vous pouvez ajuster selon vos besoins

clusters_cah = fcluster(linkage_matrix, num_clusters_cah, criterion='maxclust')

# Ajouter les labels de cluster au DataFrame
df_model['cluster_cah'] = clusters_cah

# Statistiques par cluster CAH
print("Répartition des clusters CAH:")
print(df_model['cluster_cah'].value_counts().sort_index())

print("\nStatistiques descriptives par cluster CAH:")
cluster_stats_cah = df_model.groupby('cluster_cah').mean()
cluster_stats_cah

## 3. K-Means Clustering

### 3.1 Détermination du nombre optimal de clusters avec K-Means

In [ ]:
# Méthode du coude pour K-Means
inertias = []
K_range = range(2, 11)
kmeans_settings = {"init": "random", "n_init": 10, "max_iter": 300, "random_state": 42}

for k in K_range:
    kmeans = KMeans(n_clusters=k, **kmeans_settings)
    kmeans.fit(df_scaled)
    inertias.append(kmeans.inertia_)

# Visualisation de la méthode du coude
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nombre de clusters (k)', fontsize=12)
plt.ylabel('Inertie', fontsize=12)
plt.title('Méthode du Coude pour K-Means', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

# Trouver le point optimal avec KneeLocator
knee_locator = KneeLocator(K_range, inertias, curve='convex', direction='decreasing')
optimal_k_elbow = knee_locator.elbow
print(f"Nombre optimal de clusters (méthode du coude): {optimal_k_elbow}")

In [ ]:
# Calcul des métriques d'évaluation pour K-Means
silhouette_scores_kmeans = []
calinski_harabasz_scores = []
davies_bouldin_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, **kmeans_settings)
    kmeans.fit(df_scaled)
    labels = kmeans.labels_
    
    silhouette_scores_kmeans.append(silhouette_score(df_scaled, labels))
    calinski_harabasz_scores.append(calinski_harabasz_score(df_scaled, labels))
    davies_bouldin_scores.append(davies_bouldin_score(df_scaled, labels))

# Visualisation des métriques
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].axvline(x=optimal_k_elbow, color='r', linestyle='--', label=f'Optimal k={optimal_k_elbow}')
axes[0, 0].set_xlabel('Nombre de clusters')
axes[0, 0].set_ylabel('Inertie')
axes[0, 0].set_title('Méthode du Coude')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

optimal_k_silhouette = K_range[np.argmax(silhouette_scores_kmeans)]
axes[0, 1].plot(K_range, silhouette_scores_kmeans, 'go-', linewidth=2, markersize=8)
axes[0, 1].axvline(x=optimal_k_silhouette, color='r', linestyle='--', 
                   label=f'Optimal k={optimal_k_silhouette}')
axes[0, 1].set_xlabel('Nombre de clusters')
axes[0, 1].set_ylabel('Score de Silhouette')
axes[0, 1].set_title('Score de Silhouette')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

optimal_k_ch = K_range[np.argmax(calinski_harabasz_scores)]
axes[1, 0].plot(K_range, calinski_harabasz_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 0].axvline(x=optimal_k_ch, color='r', linestyle='--', label=f'Optimal k={optimal_k_ch}')
axes[1, 0].set_xlabel('Nombre de clusters')
axes[1, 0].set_ylabel('Score de Calinski-Harabasz')
axes[1, 0].set_title('Score de Calinski-Harabasz')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

optimal_k_db = K_range[np.argmin(davies_bouldin_scores)]
axes[1, 1].plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 1].axvline(x=optimal_k_db, color='r', linestyle='--', label=f'Optimal k={optimal_k_db}')
axes[1, 1].set_xlabel('Nombre de clusters')
axes[1, 1].set_ylabel('Score de Davies-Bouldin')
axes[1, 1].set_title('Score de Davies-Bouldin')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Application de K-Means avec le nombre optimal de clusters
optimal_k = 3  # Vous pouvez ajuster selon vos besoins

kmeans_final = KMeans(n_clusters=optimal_k, **kmeans_settings)
kmeans_final.fit(df_scaled)

# Obtenir les labels de cluster
labels_kmeans = kmeans_final.labels_

# Ajouter les labels au DataFrame
df_model['cluster_kmeans'] = labels_kmeans

# Statistiques par cluster K-Means
print("Répartition des clusters K-Means:")
print(df_model['cluster_kmeans'].value_counts().sort_index())

print("\nStatistiques descriptives par cluster K-Means:")
cluster_stats_kmeans = df_model.groupby('cluster_kmeans').mean()
cluster_stats_kmeans

## 4. Comparaison des méthodes CAH et K-Means

In [ ]:
# Calcul des métriques d'évaluation pour les deux méthodes
silhouette_cah = silhouette_score(df_scaled, clusters_cah)
silhouette_kmeans = silhouette_score(df_scaled, labels_kmeans)

calinski_cah = calinski_harabasz_score(df_scaled, clusters_cah)
calinski_kmeans = calinski_harabasz_score(df_scaled, labels_kmeans)

davies_cah = davies_bouldin_score(df_scaled, clusters_cah)
davies_kmeans = davies_bouldin_score(df_scaled, labels_kmeans)

# Création d'un tableau comparatif
comparison_df = pd.DataFrame({
    'Méthode': ['CAH', 'K-Means'],
    'Score de Silhouette': [silhouette_cah, silhouette_kmeans],
    'Score de Calinski-Harabasz': [calinski_cah, calinski_kmeans],
    'Score de Davies-Bouldin': [davies_cah, davies_kmeans]
})

print("Comparaison des métriques d'évaluation:")
print(comparison_df.round(4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

important_vars = ['BALANCE', 'PURCHASES', 'CREDIT_LIMIT', 'PAYMENTS']
y_var = 'PURCHASES'


important_vars = [v for v in important_vars if v != y_var]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()


colors_cah = plt.cm.Set2(np.linspace(0, 1, df_model['cluster_cah'].nunique()))
cah_ids = sorted(df_model['cluster_cah'].unique())
cah_color = {cid: colors_cah[i] for i, cid in enumerate(cah_ids)}

colors_km = plt.cm.Set1(np.linspace(0, 1, df_model['cluster_kmeans'].nunique()))
km_ids = sorted(df_model['cluster_kmeans'].unique())
km_color = {cid: colors_km[i] for i, cid in enumerate(km_ids)}

for idx, var in enumerate(important_vars):
    ax = axes[idx]

    # --- CAH ---
    for cluster_id in cah_ids:
        cluster_data = df_model[df_model['cluster_cah'] == cluster_id]
        ax.scatter(
            cluster_data[var], cluster_data[y_var],
            marker='o', alpha=0.45, s=25,
            color=cah_color[cluster_id],
            label=f'CAH {cluster_id}'
        )

    # --- KMeans ---
    for cluster_id in km_ids:
        cluster_data = df_model[df_model['cluster_kmeans'] == cluster_id]
        ax.scatter(
            cluster_data[var], cluster_data[y_var],
            marker='^', alpha=0.45, s=25,
            color=km_color[cluster_id],
            edgecolors='black', linewidths=0.3,
            label=f'KMeans {cluster_id}'
        )

    ax.set_xlabel(var, fontsize=12)
    ax.set_ylabel(y_var, fontsize=12)
    ax.set_title(f'CAH vs K-means : {var} vs {y_var}', fontsize=12)
    ax.grid(True, alpha=0.3)

    
    ax.legend(fontsize=8, ncol=2, frameon=True)

#
for j in range(len(important_vars), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()
